# X01 · Quantization for edge inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/10_compute_deployment/X01_quantization_for_edge_inference.ipynb)

**Track:** Compute and deployment · **Skill axis (sheet):** Compute and deployment · **Time:** about 55 min

**Skill Hub concepts:** Quantization; Post-training quantization versus quantization-aware training

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("X01")

## Map: floats to small integers without losing the model

**Affine (asymmetric) quantization** to $b$ bits maps a real range $[x_{min}, x_{max}]$ onto integers $[0, 2^b - 1]$:
$$s = \frac{x_{max} - x_{min}}{2^b - 1}, \qquad z = \operatorname{round}\left(-\frac{x_{min}}{s}\right), \qquad
q = \operatorname{clip}\left(\operatorname{round}(x / s) + z,\ 0,\ 2^b - 1\right), \qquad \hat x = s\,(q - z)$$
**Symmetric** int8 (common for weights): $z = 0$, $s = \max|x| / 127$, $q \in [-127, 127]$.

| Granularity | One scale per | Error when channels differ in range |
|---|---|---|
| per-tensor | whole tensor | large: small channels lose all their resolution |
| per-channel | output channel (row) | small; standard for convolution and linear weights |

| | Post-training quantization (PTQ) | Quantization-aware training (QAT) |
|---|---|---|
| Needs | trained model + a small calibration set | training pipeline and data |
| How | measure ranges, quantize | insert fake quantization in the forward pass, train through it |
| Gradient through `round` | not needed | **straight-through estimator**: treat $\partial q / \partial x = 1$ inside the range |
| Typical int8 accuracy loss | small for big CNNs, large for small or depthwise models | recovers most of it |

Dynamic quantization stores weights in int8 and quantizes activations on the fly (good for linear and LSTM layers on CPU).

In [ ]:
import numpy as np
import torch

## Task 1: affine quantization

In [ ]:
def quant_params(xmin, xmax, bits=8):
    """Return (scale, zero_point). Extend the range to include 0 first."""
    raise NotImplementedError  # TODO

def quantize(x, scale, zp, bits=8):
    raise NotImplementedError  # TODO

def dequantize(q, scale, zp):
    raise NotImplementedError  # TODO

lab.check("affine", quant_params, quantize, dequantize)

## Task 2: per-tensor versus per-channel symmetric int8
Both return the dequantized weights (same shape as `W`) so you can measure the error.

In [ ]:
def fake_quant_per_tensor(W):
    raise NotImplementedError  # TODO

def fake_quant_per_channel(W):
    raise NotImplementedError  # TODO

lab.check("granularity", fake_quant_per_tensor, fake_quant_per_channel)

## Task 3: fake quantization with a straight-through estimator (the core of QAT)
Forward: quantize then dequantize (affine, `bits` bits). Backward: gradient 1 for inputs inside the representable
range. Trick: `x + (fq - x).detach()`.

In [ ]:
def fake_quant_ste(x, scale, zp, bits=8):
    raise NotImplementedError  # TODO

lab.check("ste", fake_quant_ste)

Post-training dynamic quantization of a small network (not graded; compare size and output).

In [ ]:
import io, warnings
net = torch.nn.Sequential(torch.nn.Linear(256, 512), torch.nn.ReLU(), torch.nn.Linear(512, 10)).eval()
def size_kb(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf); return buf.tell() / 1024
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        qnet = torch.ao.quantization.quantize_dynamic(net, {torch.nn.Linear}, dtype=torch.qint8)
    x = torch.randn(4, 256)
    print(f"float32 {size_kb(net):.0f} kB, int8 {size_kb(qnet):.0f} kB, max output difference {(net(x) - qnet(x)).abs().max():.4f}")
except (AttributeError, RuntimeError) as e:
    print("this PyTorch build has no eager dynamic quantization (moved to torchao):", e)

## Checkpoint: Math
**M1 (Apply).** Unsigned 8-bit affine quantization of the range $[-0.5, 1.5]$: zero point?

**M2 (Remember).** Storage ratio of float32 to int8 weights?

**M3 (Apply).** Scale for that range?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** A small depthwise-convolution detection model loses 9 points of accuracy after int8 PTQ. Best next step?
- a) Quantize to int4 instead
- b) Per-channel weights, better calibration data, then quantization-aware training if still short
- c) Remove the calibration step

**K2 (Understand).** Why does training need the straight-through estimator, and why does QAT recover accuracy that
PTQ loses?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `x01.h`).
3. In `roboprep/checks/x01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()